# Notebook 03: Exploratory Data Analysis & Visualization

**Project:** City of Boston – Building & Housing Violations Analysis  
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Overview

This notebook explores the cleaned violations dataset through four lenses:
1. **Violation type frequency** — bar charts of top violation codes and descriptions
2. **Neighborhood breakdown** — which areas have the most violations
3. **Temporal trends** — yearly and monthly patterns
4. **Spatial visualization** — choropleth map and coordinate heatmap

All figures are saved to the `figures/` directory.

In [14]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for saving figures
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROJECT_ROOT = Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / 'violations_clean.csv', low_memory=False)
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
print(f'Loaded {len(df):,} records')
df.head(2)

Loaded 16,914 records


,case_no,status,status_dttm,year,month,day_of_week,code,description,neighborhood,violation_street,violation_zip,ward,latitude,longitude,has_coords,has_neighborhood
0,V897990,Open,2026-03-20 15:27:46,2026.0,3.0,Friday,107.4,Failed to comply w permit term,Dorchester,Elder,02125,07,42.32023,-71.06334,True,True
1,V897956,Open,2026-03-20 10:37:20,2026.0,3.0,Friday,102.8,Maintenance,Dorchester,Quincy,02121,13,42.31383,-71.07762,True,True


## 1. Top Violation Types

We identify the 20 most frequent violation descriptions across all neighborhoods and years. Understanding which violation types are most common informs priorities for housing code enforcement.

In [15]:
top_desc = df['description'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
colors = sns.color_palette('Blues_r', len(top_desc))
bars = ax.barh(top_desc.index[::-1], top_desc.values[::-1], color=colors)
ax.set_xlabel('Number of Violations')
ax.set_title('Top 20 Violation Types – Boston Building & Property Violations', fontsize=13)
for bar, val in zip(bars, top_desc.values[::-1]):
    ax.text(val + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '01_top_violation_types.png', bbox_inches='tight')
plt.close()
print('Saved: 01_top_violation_types.png')
top_desc

Saved: 01_top_violation_types.png


description
Failure to Obtain Permit          4189
Unsafe and Dangerous              3622
Maintenance                       1730
Testing & Certification            922
Unsafe Structures                  744
Right of Entry                     687
Inspections                        515
Failed to comply w permit term     468
Certificate of Occupancy           305
Building or Use of Premise req     288
Protection of Adj. Property        267
Unknown                            247
Failure to secure permit           233
Unsafe Maintenance                 221
Failure to Secure Permit           189
Periodic Inspections               177
Unsafe Structure                    97
No use of premises permit:          90
Unsafe & Dangerous                  86
Means of Egress                     78
Name: count, dtype: int64

## 2. Violations by Neighborhood

A bar chart of total violations per neighborhood

In [16]:
nbhd_counts = df[df['has_neighborhood'] == True]['neighborhood'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
palette = sns.color_palette('Spectral', len(nbhd_counts))
ax.bar(nbhd_counts.index, nbhd_counts.values, color=palette)
ax.set_xticklabels(nbhd_counts.index, rotation=45, ha='right')
ax.set_ylabel('Violation Count')
ax.set_title('Total Building & Property Violations by Neighborhood', fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '02_violations_by_neighborhood.png', bbox_inches='tight')
plt.close()
print('Saved: 02_violations_by_neighborhood.png')
nbhd_counts

Saved: 02_violations_by_neighborhood.png


C:\Users\Jiacheng He\AppData\Local\Temp\ipykernel_31564\2272034768.py:6: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(nbhd_counts.index, rotation=45, ha='right')


neighborhood
Dorchester       4602
Downtown         2761
East Boston      1761
Roxbury          1620
Mattapan          877
South Boston      865
Hyde Park         831
Brighton          753
Allston           608
Roslindale        545
Jamaica Plain     469
Charlestown       419
West Roxbury      382
Mission Hill      363
South End           4
West End            2
Chinatown           1
Fenway              1
Back Bay            1
Name: count, dtype: int64

## 3. Temporal Trends

### 3a. Violations Per Year

The annual count of violations shows whether enforcement activity is increasing or decreasing over time. Note that records without a `status_dttm` are excluded from this analysis.

In [17]:
df_dated = df[df['status_dttm'].notna() & df['year'].notna()].copy()
df_dated['year'] = df_dated['year'].astype(int)

year_counts = df_dated.groupby('year').size()
# Filter to years with sufficient data (at least 12 months represented)
year_counts = year_counts[year_counts.index >= 2010]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(year_counts.index, year_counts.values, marker='o', linewidth=2.5,
        color='steelblue', markersize=7)
ax.fill_between(year_counts.index, year_counts.values, alpha=0.15, color='steelblue')
ax.set_xlabel('Year')
ax.set_ylabel('Violation Count')
ax.set_title('Annual Building & Property Violations in Boston', fontsize=13)
ax.set_xticks(year_counts.index)
ax.set_xticklabels(year_counts.index, rotation=45)
for x, y in zip(year_counts.index, year_counts.values):
    ax.annotate(f'{y:,}', (x, y), textcoords='offset points',
                xytext=(0, 8), ha='center', fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '03_violations_per_year.png', bbox_inches='tight')
plt.close()
print('Saved: 03_violations_per_year.png')
print(year_counts)

Saved: 03_violations_per_year.png
year
2010    1450
2011    1148
2012    1454
2013    1437
2014    1443
2015    1669
2016     946
2017     954
2018     784
2019     933
2020     731
2021     687
2022     903
2023     657
2024     846
2025     708
2026     153
dtype: int64


### 3b. Monthly Seasonality

We examine whether violations are filed more in certain months, which could reflect seasonal building inspection cycles or tenant-driven patterns.

In [18]:
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df_dated.groupby('month').size().reindex(range(1,13), fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1,13), monthly.values, color=sns.color_palette('coolwarm', 12))
ax.set_xticks(range(1,13))
ax.set_xticklabels(month_names)
ax.set_ylabel('Violation Count')
ax.set_title('Monthly Seasonality of Building Violations (All Years)', fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '04_monthly_seasonality.png', bbox_inches='tight')
plt.close()
print('Saved: 04_monthly_seasonality.png')

Saved: 04_monthly_seasonality.png


### 3c. Heatmap: Neighborhood × Year

This heatmap reveals how violation patterns in each neighborhood have changed over the years.

In [19]:
df_nbhd_dated = df_dated[df_dated['has_neighborhood'] == True].copy()
pivot = df_nbhd_dated.groupby(['neighborhood', 'year']).size().unstack(fill_value=0)

# Keep only years 2010+ and neighborhoods with meaningful data
pivot = pivot.loc[:, pivot.columns >= 2010]
pivot = pivot[pivot.sum(axis=1) >= 10]  # at least 10 total violations

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, ax=ax,
            cbar_kws={'label': 'Violation Count'})
ax.set_title('Violations by Neighborhood and Year (Heatmap)', fontsize=13)
ax.set_xlabel('Year')
ax.set_ylabel('Neighborhood')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '05_neighborhood_year_heatmap.png', bbox_inches='tight')
plt.close()
print('Saved: 05_neighborhood_year_heatmap.png')

Saved: 05_neighborhood_year_heatmap.png


## 4. Top Violation Types by Neighborhood

A stacked bar chart shows the composition of violation types per neighborhood, revealing whether neighborhoods face different dominant violation categories.

In [20]:
top5_types = df['description'].value_counts().head(5).index.tolist()
df_top = df[df['has_neighborhood'] == True].copy()
df_top['desc_grouped'] = df_top['description'].where(
    df_top['description'].isin(top5_types), other='Other'
)

stacked = df_top.groupby(['neighborhood', 'desc_grouped']).size().unstack(fill_value=0)
# Sort by total
stacked = stacked.loc[stacked.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(14, 7))
stacked.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')
ax.set_xlabel('Neighborhood')
ax.set_ylabel('Number of Violations')
ax.set_title('Top Violation Types by Neighborhood (Stacked)', fontsize=13)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '06_violation_types_by_neighborhood.png', bbox_inches='tight')
plt.close()
print('Saved: 06_violation_types_by_neighborhood.png')

Saved: 06_violation_types_by_neighborhood.png


## 5. Spatial Scatter Map

We plot violation locations using their latitude/longitude coordinates. Each point is colored by violation status (Open vs. Closed) to reveal spatial patterns in unresolved violations.

In [21]:
df_geo = df[df['has_coords'] == True].copy()
# Filter to Boston bounding box
df_geo = df_geo[
    (df_geo['latitude'].between(42.22, 42.40)) &
    (df_geo['longitude'].between(-71.19, -70.99))
]
print(f'Records with valid coordinates in Boston bbox: {len(df_geo):,}')

status_colors = {'Open': '#e74c3c', 'Closed': '#2980b9'}
fig, ax = plt.subplots(figsize=(10, 10))
for status, group in df_geo.groupby('status'):
    color = status_colors.get(status, '#7f8c8d')
    ax.scatter(group['longitude'], group['latitude'],
               s=4, alpha=0.4, color=color, label=status)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Geographic Distribution of Boston Building Violations', fontsize=13)
ax.legend(markerscale=5, title='Status')
plt.tight_layout()
fig.savefig(FIGURES_DIR / '07_spatial_scatter.png', bbox_inches='tight')
plt.close()
print('Saved: 07_spatial_scatter.png')

Records with valid coordinates in Boston bbox: 16,700
Saved: 07_spatial_scatter.png


## 6. Violation Status Distribution

A pie chart summarizing the proportion of Open vs. Closed violations gives a quick snapshot of compliance rates.

In [24]:
fig, ax = plt.subplots(figsize=(8, 7))
wedges, texts, autotexts = ax.pie(
    status_counts.values,
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=1.15,
    colors=['#e74c3c', '#2ecc71', '#3498db', '#f39c12'][:len(status_counts)]
)
ax.legend(wedges, status_counts.index,
          title='Status',
          loc='upper left',
          bbox_to_anchor=(1, 1))
ax.set_title('Violation Status Distribution', fontsize=13)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '08_status_distribution.png', bbox_inches='tight')
plt.close()

## Summary

Key findings from EDA:
- **Maintenance** and **Unsafe/Dangerous** are the most common violation types citywide.
- **Dorchester, Roxbury, and East Boston** have the highest raw violation counts.
- Violations show mild seasonal patterns, with slightly higher activity in spring/summer months.
- The heatmap reveals persistent hotspots in certain neighborhoods across multiple years.
- A substantial fraction of violations remain **Open**, indicating unresolved compliance issues.